# 03 - Image-permutation control on ViProKG (1 GPU, about 0.3 GPU-hours)

Question: on ViProKG, does the *content* of the images add signal, or only the presence of an image channel?
Two branches with exactly the same code and flags as the signed headline run MENTOR_base+L (lambda_align 8,
`--use_image 1`, dim 400, lr 5e-4, g = 0.15), three seeds {42, 7, 2024}, one run per branch and seed (6 runs):

* **image_real** - the real image features `V.pt`;
* **image_permuted** - the rows of `V.pt` are permuted among the entities that have an image (7,790 of 8,255); the
  permutation is fixed per seed (blake2b-keyed shuffle, so it is identical in every process), the has-image mask is
  unchanged, and the sha256 of every permutation is logged and checked against the value computed offline.

The branch with images switched off already exists and is not re-run. Both branches of a seed share seed and data,
so differences are paired.

**Add Data (2 datasets, internet may stay OFF):** (1) the release dataset (folder containing `scripts/verify_tables.py`
and `data/experimental/v3/ext/ViMMKG2`); (2) a PhoBERT checkpoint dataset (a folder whose path contains "phobert" and
that holds `config.json` plus `pytorch_model.bin` or `model.safetensors`); it is only used to build `T_ctx.pt`.

## Pre-specified criterion (written before any run cell executes)

**|delta overall (real - permuted)| <= 0.65 MRR points** (3-seed mean, test, both directions).

* PASS -> the image content adds no detectable signal on ViProKG (permuted images do as well as real ones).
* FAIL -> the image content matters on ViProKG.

Reported without thresholds: delta by group (overall / ordinary / slice) x (all / mention-overlap / mention-free) with the
95% across-seed CI (t, df = 2) and a query-level paired bootstrap CI (B = 1000). Informational, not a gate: the level of
the real branch against the signed MENTOR_base+L total MRR 41.20 (within 0.65 expected).
Design note: in the earlier MKG-W / MKG-Y control the rows of the image matrix were shuffled across all entities; here
only image-bearing entities are permuted and the mask is fixed, so entities without an image stay without.

In [ ]:
print("""PRE-SPECIFIED (written before the run cells):
  |delta overall (real - permuted)| <= 0.65  ->  image content adds no detectable signal on ViProKG
  group deltas (overall/ordinary/slice x all/overlap/free): reported, with CIs, no thresholds
  informational: real-branch level vs signed 41.20 (expected within 0.65)""")

In [ ]:
import csv, glob, hashlib, json, os, random, shutil, statistics as st, subprocess, sys, time
INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")
OUT_ROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/tmp/mentor_nb3"
SMOKE = bool(os.environ.get("ANH_V2_SMOKE"))   # local test switch only; not used on Kaggle

def run_streaming(cmd, cwd=None, env=None, tail=40):
    """Run a command, print its output line by line, return (returncode, last lines)."""
    p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    last = []
    for line in p.stdout:
        print(line, end="", flush=True)
        last.append(line); last = last[-tail:]
    return p.wait(), last

hits = sorted(glob.glob(f"{INPUT}/**/scripts/verify_tables.py", recursive=True))
assert hits, "Attach the release dataset (the folder containing scripts/verify_tables.py)."
REL = os.path.dirname(os.path.dirname(hits[0]))
EXT = f"{REL}/data/experimental/v3/ext/ViMMKG2"
EDGES = f"{REL}/data/experimental/v3/canh_v2/v10_canh_nhac_ten_ViMMKG2_v2.tsv"
assert hashlib.sha256(open(EDGES, "rb").read()).hexdigest()[:16] == "cefc05167b7123d3", "mention-edge file differs from the signed one"
PHOBERT = next((dp for dp, _, fns in sorted(os.walk(INPUT))
                if "config.json" in fns and ("pytorch_model.bin" in fns or "model.safetensors" in fns) and "phobert" in dp.lower()), None)
print("release:", REL, "\nphobert:", PHOBERT)

In [ ]:
W = f"{OUT_ROOT}/mentor"; D = f"{W}/data/ViMMKG2"; RES = f"{W}/result_anh_v2"
os.makedirs(D, exist_ok=True); os.makedirs(RES, exist_ok=True)
for f in os.listdir(f"{REL}/src"):
    if f.endswith(".py"): shutil.copy(f"{REL}/src/{f}", f"{W}/{f}")
for f in os.listdir(EXT):
    if f.endswith((".pt", ".txt", ".tsv", ".json")): shutil.copy(f"{EXT}/{f}", f"{D}/{f}")
def sha16(*ps):
    h = hashlib.sha256()
    for p in sorted(ps): h.update(open(p, "rb").read())
    return h.hexdigest()[:16]
import torch
N_ENT = sum(1 for _ in open(f"{D}/entities.txt", encoding="utf-8"))
M = torch.load(f"{D}/V_mask.pt", map_location="cpu")           # True = entity has NO image
HAS = (~M).view(-1).nonzero().view(-1)                          # ids of entities that have an image
print("entities", N_ENT, "| with image", len(HAS), "| inputs sha:", sha16(f"{D}/V.pt", f"{D}/V_mask.pt", f"{D}/L_feat.pt"))
if not SMOKE:
    assert N_ENT == 8255 and len(HAS) == 7790, "unexpected ViProKG files"

## 1. Build the text tensor `T_ctx.pt` (PhoBERT mean pooling; released code)

In [ ]:
sys.path.insert(0, W)
from build_text_context import build_text_context, load_encoder
if not os.path.exists(f"{D}/T_ctx.pt"):
    assert PHOBERT, "Attach a PhoBERT checkpoint dataset (path containing 'phobert')."
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok, model = load_encoder(PHOBERT, device)
    _ids = json.load(open(f"{D}/ViMMKG2-textual-phobert-seg.json"))["0"]
    print("decode check (entity 0, should read as Vietnamese text):", tok.decode([x for x in _ids if x < tok.vocab_size], skip_special_tokens=True)[:150])
    T, _ = build_text_context("ViMMKG2", D, f"{D}/ViMMKG2-textual-phobert-seg.json", tok, model, batch=64, device=device)
    torch.save(T, f"{D}/T_ctx.pt")
    del model
T = torch.load(f"{D}/T_ctx.pt", map_location="cpu")
nz = int((T.abs().sum(1) > 0).sum())
print("T_ctx", tuple(T.shape), "rows with text:", nz, "| sha16:", sha16(f"{D}/T_ctx.pt"), "(headline run used a9b350f5cf651d7c; informational)")
if not SMOKE:
    assert tuple(T.shape) == (8255, 768) and nz == 8207, "T_ctx build did not give the expected shape / coverage"

## 2. Fixed image permutations (per seed) and their checks

In [ ]:
SEEDS = [int(x) for x in os.environ.get("ANH_V2_SEEDS", "42,7,2024").split(",")]
EXPECTED_SHA = {42: "355356b66fadec07", 7: "5dee4c1281edf82f", 2024: "b3f116f1b0ccda05"}  # computed offline, another process
V = torch.load(f"{D}/V.pt", map_location="cpu")
PERM_LOG = {}
def make_permuted(seed):
    rng = random.Random(int(hashlib.blake2b(f"ViMMKG2|image_permutation|{seed}".encode(), digest_size=8).hexdigest(), 16))
    order = list(range(len(HAS))); rng.shuffle(order)
    src = HAS[torch.tensor(order)]                      # entity HAS[i] receives the image of entity src[i]
    Vp = V.clone(); Vp[HAS] = V[src]
    assert torch.equal(Vp[M.view(-1)], V[M.view(-1)]), "entities without an image must stay unchanged"
    sha = hashlib.sha256(src.numpy().astype("int64").tobytes()).hexdigest()[:16]
    fixed = int((src == HAS).sum())
    changed = int((Vp.view(len(V), -1) != V.view(len(V), -1)).any(1).sum())
    torch.save(Vp, f"{D}/V_perm_s{seed}.pt"); torch.save(M.clone(), f"{D}/V_perm_s{seed}_mask.pt")
    PERM_LOG[seed] = {"sha16": sha, "fixed_points": fixed, "rows_changed": changed, "n_image_entities": len(HAS)}
    print(f"seed {seed}: permutation sha16 {sha} | fixed points {fixed} | image rows changed {changed}/{len(HAS)}")
    if not SMOKE:
        assert sha == EXPECTED_SHA.get(seed, sha), "permutation differs from the offline value (non-deterministic shuffle?)"
        assert fixed == 0 and changed >= 0.99 * len(HAS)
for s in SEEDS: make_permuted(s)
json.dump(PERM_LOG, open(f"{RES}/permutation_log.json", "w"), indent=1)

## 3. Train: 3 seeds x {image_real, image_permuted} (same flags as the signed MENTOR_base+L run, lambda 8)

In [ ]:
EPOCHS = os.environ.get("MENTOR_EPOCHS", "150"); VALID_EVERY = os.environ.get("MENTOR_VALID_EVERY", "5")
BUDGET_MIN = 90.0
def cmd_for(tag, seed):
    vpath = f"{D}/V.pt" if tag == "real" else f"{D}/V_perm_s{seed}.pt"
    return [sys.executable, "-u", "train.py", "--data_dir", D, "--t_path", f"{D}/T_ctx.pt", "--v_path", vpath,
            "--use_text", "1", "--use_image", "1", "--variant", "LSR", "--lsr_L", f"{D}/L_feat.pt",
            "--lsr_off", "RM", "--lsr_l1", "1",
            "--dump_full", f"{RES}/dump_test_{tag}_s{seed}.tsv", "--dump_full_valid", f"{RES}/dump_valid_{tag}_s{seed}.tsv",
            "--num_epoch", EPOCHS, "--valid_epoch", VALID_EVERY, "--lr", "0.0005", "--batch_size", "512", "--dim", "400",
            "--dim_rel", "64", "--reciprocal", "1", "--entity_bias", "0", "--text_mode", "mean", "--gate_mode", "gate_off",
            "--lambda_align", "8.0", "--seed", str(seed), "--exp", f"anh_v2_{tag}_s{seed}", "--scores_g", "0.15"]
def zipit():
    if os.path.isdir("/kaggle/working"):
        subprocess.run(["bash", "-c", f"cd {W} && zip -qrX /kaggle/working/anh_v2_out.zip result_anh_v2 -x '*.pt'"], check=False)
t_all = time.time()
env = dict(os.environ, VIS_DIM="32")
for s in SEEDS:
    for tag in ("real", "perm"):
        if os.path.exists(f"{RES}/dump_test_{tag}_s{s}.tsv"):
            print(f"[{tag} s{s}] already done, skipped"); continue
        if (time.time() - t_all) / 60 > BUDGET_MIN:
            print("time budget reached; results so far are saved"); zipit(); raise SystemExit
        c = cmd_for(tag, s); t0 = time.time()
        rc, last = run_streaming(c, cwd=W, env=env)
        print(f"\n[{tag} s{s}] rc={rc} {(time.time() - t0) / 60:.1f} min", flush=True)
        assert rc == 0, f"training failed ({tag}, seed {s}); see the output above"
        json.dump({"cmd": c, "seed": s, "tag": tag, "train_py_sha16": sha16(f"{W}/train.py"), "T_ctx_sha16": sha16(f"{D}/T_ctx.pt"),
                   "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
                   "permutation": PERM_LOG.get(s) if tag == "perm" else None},
                  open(f"{RES}/config_{tag}_s{s}.json", "w"), indent=1)
        zipit()
print(f"all runs done in {(time.time() - t_all) / 60:.1f} min")

## 4. Score: paired delta (real - permuted), CIs, and the pre-specified criterion

In [ ]:
rc, _ = run_streaming([sys.executable, f"{REL}/scripts/score_anh_v2.py", RES, "--ext", D, "--edges", EDGES,
                       "--out_json", f"{RES}/score_anh_v2.json"])
assert rc == 0
res = json.load(open(f"{RES}/score_anh_v2.json"))
print("\nVERDICT (pre-specified |delta overall| <= 0.65):", "PASS" if res["criterion"]["pass"] else "FAIL",
      f"(delta overall = {res['criterion']['delta_overall']:+.3f})")
il = res["informational_real_level"]
print(f"informational: real branch {il['mrr_real']:.2f} vs signed {il['signed_value']:.2f} (diff {il['diff']:+.2f}; within 0.65: {il['within_0.65']})")
zipit()